# 3. Data Preparation: Rekayasa 75 Fitur Invarian Lantai & Pembagian Data Berbasis Kelompok

Tahap **Data Preparation** menyiapkan dataset citra dari **`2-data_understanding/data/`** agar siap dimodelkan secara ilmiah tanpa kebocoran data (*0% Data Leakage*) dan tahan terhadap *overfitting*:

1. **Koreksi Orientasi EXIF & Normalisasi Ukuran ($224 \times 224$):** Citra dirotasi otomatis sesuai metadata EXIF lalu di-*resize* langsung ke $224 \times 224$ piksel tanpa padding abu-abu agar tidak menimbulkan garis tepi buatan di pinggir gambar.
2. **Deduplikasi Berbasis Hash SHA-256 & Koreksi Label:** Mengidentifikasi dan mengecualikan **3 citra duplikat identik** serta menerapkan 4 koreksi label terverifikasi sehingga tersisa **150 citra unik**.
3. **Ekstraksi 75 Fitur Visual Tahan Lantai (*Floor-Invariant Feature Engineering*):**
   * **Keragaman Warna Klasik (13 Fitur):** Standar deviasi RGB & HSV (`std_red`, `std_green`, `std_blue`, `std_overall`, `hue_std`, `sat_mean`, `sat_std`, `val_std`), warna oponen ($R-G$, $Y-B$), dan variansi warna blok $4 \times 4$. Nilai *mean RGB* global sengaja tidak dijadikan fitur model agar model tidak *overfitting* terhadap warna lantai kelompok tertentu.
   * **Keteraturan Orientasi & Tensor Struktur (12 Fitur):** Entropi & koefisien Gini arah gradien Sobel, rasio tepi ortogonal vs diagonal, koherensi tensor struktur global/blok, serta koherensi area tengah (`center_coh`).
   * **Sebaran Kepadatan Tepi Spasial (7 Fitur):** Kepadatan tepi pada grid $4 \times 4$ dan $8 \times 8$ (entropi, koefisien variasi CV, Gini).
   * **Spektrum Frekuensi 2D FFT (9 Fitur):** Energi 6 pita frekuensi radial serta konsentrasi sudut (*FFT Angular Entropy & Gini*) untuk mendeteksi pola barisan sejajar.
   * **Profil Proyeksi, Radon Rotated Alignment, & Simetri (17 Fitur):** Variansi proyeksi baris/kolom, proyeksi Radon multi-sudut (*rotation-invariant alignment*), kesimetrian cermin kiri-kanan & atas-bawah, serta rasio pusat-vs-tepi dan dalam-vs-luar (`inner_outer_ratio`).
   * **Geometri & Sebaran Objek *Foreground* (9 Fitur):** Jarak radial titik tepi terhadap pusat massa (`fg_dist_mean`, `fg_dist_p75`, `fg_dist_iqr`), dispersi diagonal (`fg_spread_diag`), elongasi PCA (`fg_elongation`), dan jumlah komponen.
   * **Tekstur Lokal LBP & GLCM (8 Fitur):** *Local Binary Patterns* dan *Gray-Level Co-occurrence Matrix* ($0^\circ, 45^\circ, 90^\circ, 135^\circ$).
4. **Pembagian Data Berbasis Kelompok (*Group-Independent Split*):** **Kelompok 2** (15 Rapi + 15 Berantakan = **30 citra**) dipisahkan secara mutlak sebagai **Unseen Test Set**. Kelompok lainnya (**120 citra unik**) masuk **Development Set**.
5. **Augmentasi Pencerminan Horizontal Khusus Data Latih:** Versi cermin horizontal (*mirror*) diekstrak **khusus untuk data pengembangan**, sehingga saat pelatihan tiap fold dapat menggandakan sampel latih tanpa menyentuh validasi atau data uji.


In [1]:
import sys
from pathlib import Path
import pandas as pd


def find_project_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "src" / "10-project_citra"):
            if (candidate / "3-data_preparation").is_dir() and (candidate / "6-deployment").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Folder proyek 10-project_citra tidak ditemukan dari direktori kerja.")


PROJECT_DIR = find_project_dir()
DATASET_DIR = PROJECT_DIR / "2-data_understanding" / "data"
if not all((DATASET_DIR / name).is_dir() for name in ("RAPI", "BERANTAKAN")):
    raise FileNotFoundError(f"Folder kelas dataset tidak lengkap di: {DATASET_DIR}")

sys.path.insert(0, str(PROJECT_DIR))
from feature_extraction import FEATURE_COLUMNS
from pipeline import prepare_dataset

PREPARATION_DIR = PROJECT_DIR / "3-data_preparation"
PROCESSED_DIR = PREPARATION_DIR / "data" / "processed"
print(f"Dataset sumber : {DATASET_DIR}")
print(f"Jumlah fitur pemodelan yang diekstrak: {len(FEATURE_COLUMNS)} fitur")


Dataset sumber : D:\university\semester 5_book\proyek sain data\src\10-project_citra\2-data_understanding\data
Jumlah fitur pemodelan yang diekstrak: 75 fitur


---
## 3.1 Eksekusi Ekstraksi 75 Fitur, Deduplikasi, & Penyiapan Manifest


In [2]:
df_all = prepare_dataset()
print("\nContoh 5 baris pertama manifest dataset:")
display(df_all[["filename", "kelompok", "class", "include_in_model", "split"] + FEATURE_COLUMNS[:6]].head(5))


Dataset: 153 citra, 150 citra unik dipakai; fitur pemodelan: 75
Koreksi label: 4; duplikat dikecualikan: 3
class  Berantakan  Rapi
split                  
test           15    15
train          66    54
Train: semua kelompok kecuali kelompok 2; Test (Unseen): kelompok 2
Manifest tersimpan: D:\university\semester 5_book\proyek sain data\src\10-project_citra\3-data_preparation\dataset_manifest.csv

Contoh 5 baris pertama manifest dataset:


,filename,kelompok,class,include_in_model,split,std_red,std_green,std_blue,std_overall,hue_std,sat_mean
0,berantakan 1.jpeg,kelompok 1,Berantakan,True,train,61.5592,62.7991,61.9206,62.0929,49.7654,55.5316
1,berantakan 2.jpeg,kelompok 1,Berantakan,True,train,55.9514,61.8468,66.5182,61.4388,52.8766,64.7301
2,WhatsApp Image 2026-11-08 at 17.44.49.jpeg,kelompok 1,Rapi,True,train,53.4744,55.0807,53.8657,54.1403,61.9107,45.1033
3,WhatsApp Image 2026-11-08 at 17.44.50.jpeg,kelompok 1,Rapi,True,train,46.7980,51.5254,54.6171,50.9802,47.9568,57.8471
4,WhatsApp Image 2026-11-08 at 17.44.52.jpeg,kelompok 1,Rapi,True,train,50.3734,57.2579,63.7871,57.1395,57.4631,63.9910


---
## 3.2 Verifikasi Pemisahan *Group-Wise* & Uji Signifikansi Fitur (ANOVA F-Test pada Development Set)


In [3]:
from sklearn.feature_selection import f_classif

eligible = df_all.loc[df_all["include_in_model"]].reset_index(drop=True)
train_df = eligible.loc[eligible["split"] == "train"].reset_index(drop=True)
test_df = eligible.loc[eligible["split"] == "test"].reset_index(drop=True)

assert len(set(train_df["kelompok"]) & set(test_df["kelompok"])) == 0, "Terjadi kebocoran kelompok!"

print("=== Rekapitulasi Pembagian Dataset Unik ===")
display(eligible.groupby(["split", "class"]).size().unstack(fill_value=0))
print(f"Development Set (Train + CV) : {len(train_df)} citra unik dari {train_df['kelompok'].nunique()} kelompok")
print(f"Unseen Test Set (Kelompok 2) : {len(test_df)} citra unik dari {test_df['kelompok'].nunique()} kelompok")

X_dev = train_df[FEATURE_COLUMNS].to_numpy()
y_dev = train_df["label"].to_numpy()
f_scores, p_values = f_classif(X_dev, y_dev)
df_f = pd.DataFrame({
    "Fitur": FEATURE_COLUMNS,
    "ANOVA_F": f_scores,
    "p_value": p_values,
}).sort_values("ANOVA_F", ascending=False).reset_index(drop=True)

print("\nTop 15 Fitur Paling Informatif (Dihitung Murni pada Development Set):")
display(df_f.head(15).round(5))


=== Rekapitulasi Pembagian Dataset Unik ===


class,Berantakan,Rapi
split,,
test,15,15
train,66,54


Development Set (Train + CV) : 120 citra unik dari 6 kelompok
Unseen Test Set (Kelompok 2) : 30 citra unik dari 1 kelompok

Top 15 Fitur Paling Informatif (Dihitung Murni pada Development Set):


,Fitur,ANOVA_F,p_value
0,fg_dist_p75,37.04609,0.00000
1,fg_spread_diag,36.97184,0.00000
2,fg_dist_mean,36.78970,0.00000
3,hue_std,18.86805,0.00003
4,fft_band_1,18.86116,0.00003
5,ed_cv,17.53161,0.00005
6,ed_gini,16.72158,0.00008
7,lr_edge_sym,16.55853,0.00009
8,center_border_ratio,16.48950,0.00009
9,spatial_edge_entropy,15.64225,0.00013


---
## 3.3 Kesimpulan Tahap Data Preparation

1. **Bebas Kebocoran Data:** Sebanyak 3 duplikat identik disaring sehingga tersisa **150 citra unik**, dan Kelompok 2 terisolasi penuh sebagai *Unseen Test Set*.
2. **Daya Pembeda Geometri & Spektral:** Uji statistik ANOVA F-Score pada data pengembangan membuktikan bahwa fitur sebaran objek *foreground* (`fg_dist_p75`, `fg_spread_diag`, `fg_dist_mean`), keragaman warna (`hue_std`), spektrum 2D FFT (`fft_band_1`, `fft_ang_entropy`), variasi kepadatan tepi (`ed_cv`, `ed_gini`), dan rasio dalam-vs-luar (`inner_outer_ratio`) memiliki signifikansi sangat tinggi ($p < 0,0003$).
